|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 0:</h2>|<h1>From a Program to a Model<h1>|
|<h2>Section:</h2>|<h1>Attention<h1>|
|<h2>Lecture:</h2>|<h1><b>Code challenge: memoize the lookup<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import matplotlib_inline.backend_inline
matplotlib_inline.backend_inline.set_matplotlib_formats('svg')

rng = np.random.default_rng(0)

def softmax(scores, axis=-1):
  exponentials = np.exp(scores - scores.max(axis=axis, keepdims=True))
  return exponentials / exponentials.sum(axis=axis, keepdims=True)
import torch

Write attention two ways: over the whole text, and one token at a time with a
KV cache. Then prove that the two give the same numbers, and count the work
that the cache saves. This challenge needs no GPU.

In [ ]:
### run this cell
NUM_TOKENS, HIDDEN = 10, 16
hidden_states = rng.normal(size=(NUM_TOKENS, HIDDEN))
QUERY_WEIGHT, KEY_WEIGHT, VALUE_WEIGHT = (rng.normal(size=(HIDDEN, HIDDEN)) / np.sqrt(HIDDEN)
                                          for _ in range(3))
print(f'{NUM_TOKENS} tokens, hidden size {HIDDEN}')

# Exercise 1: attention over the whole text

Compute the queries, keys and values of all tokens. Score each query against
each key, and scale by the square root of `HIDDEN`. Apply the causal mask.
Then softmax, and mix the values.

In [ ]:
def causal_attention(hidden_states):
  """(tokens, HIDDEN) -> (tokens, HIDDEN): the output of each token."""
  

outputs = causal_attention(hidden_states)
# Check against the attention function of torch.
as_tensor = lambda array: torch.tensor(array)[None]
expected = torch.nn.functional.scaled_dot_product_attention(
    as_tensor(hidden_states @ QUERY_WEIGHT), as_tensor(hidden_states @ KEY_WEIGHT),
    as_tensor(hidden_states @ VALUE_WEIGHT), is_causal=True)[0].numpy()
print('the same as torch:', np.allclose(outputs, expected))

# Exercise 2: one token at a time, with a cache

`step()` receives the hidden state of ONE new token. Append its key and its
value to the cache. Then do its lookup over all the keys and values in the
cache. Do not compute the key or value of an earlier token again.

In [ ]:
class CachedAttention:
  """Attention for one new token at a time. It keeps a KV cache."""
  def __init__(self):
    self.keys = []            # one key for each token so far
    self.values = []          # one value for each token so far

  def step(self, hidden_state):
    """(HIDDEN,) -> (HIDDEN,): the output of the new token."""
    

attention = CachedAttention()
step_outputs = np.array([attention.step(hidden_state) for hidden_state in hidden_states])
print('one token at a time == the whole text:', np.allclose(step_outputs, outputs))
print('tokens in the cache:', len(attention.keys))

# Exercise 3: count the work

Count the multiply-adds of one attention layer, to generate `new_tokens`
tokens after a prompt. A projection of one token costs `HIDDEN * HIDDEN`. A
lookup over `length` tokens costs `2 * length * HIDDEN`: the scores, then the
mix.

- Without a cache, each step runs a full pass over the whole text so far.
- With a cache, the prefill runs one full pass over the prompt. Each decode
  step then projects one token, and does one lookup over the context.

In [ ]:
def work_without_cache(prompt_tokens, new_tokens):
  

def work_with_cache(prompt_tokens, new_tokens):
  

PROMPT_TOKENS = 100
lengths = [10, 100, 1000, 4000]
for new_tokens in lengths:
  ratio = work_without_cache(PROMPT_TOKENS, new_tokens) / work_with_cache(PROMPT_TOKENS, new_tokens)
  print(f'{new_tokens:>5} new tokens: the cache does {ratio:6.1f}x less work')

### Before you open the solution

1. In Exercise 3 the saving grows with the length. Is it linear? Which term of
   your formula decides that?
2. Your cache in Exercise 2 is a Python list that grows. A GPU wants memory of
   a fixed size, reserved in advance. What goes wrong if each sequence
   reserves memory for the longest possible reply? Part 3 answers this.
3. Exercise 1 and Exercise 2 agree to about 1e-15. With bf16 numbers, they
   agree only to about 1e-2. Why can the order of the additions change the
   result?